# HALD → Knowledge Graph (KG) Builder

**Source:** HALD (Human Aging and Longevity Database) — `Entities.csv` + `Roles.csv`  
**Species:** *Homo sapiens*

## What this notebook does

1. Loads HALD entity and triple files, maps IDs to names and types.
2. Normalises entity types: Carbohydrate/Lipid/Peptide/Protein → `ChemicalEntity`, RNA → `Gene`.
3. Removes `Toxin` and `Pharmaceutical Preparations` edge types.
4. For each of the **13 relation types**, resolves Head/Tail identifiers to canonical ontology IDs:
   - **Chemical** nodes: DrugBank ID → PubChem CID (case-insensitive)
   - **Disease** nodes: disease name → DOID (via DO label lookup) → MeSH ID (fallback)
   - **Gene** nodes: symbol → NCBI description (validation)
   - **Mutation** nodes: passed through as-is (no external ID mapping)
5. Exports one CSV per relation type.

## Output files (16 total)
```
HALD/   Chemical_Chemical_HALD.csv      HALD/   Disease_Chemical_HALD.csv
HALD/   Chemical_Disease_HALD.csv       HALD/   Disease_Disease_HALD.csv
HALD/   Chemical_Gene_HALD.csv          HALD/   Disease_Gene_HALD.csv
HALD/   Chemical_Mutation_HALD.csv      HALD/   Disease_Mutation_HALD.csv
HALD/   Gene_Chemical_HALD.csv          HALD/   Mutation_Chemical_HALD.csv
HALD/   Gene_Disease_HALD.csv           HALD/   Mutation_Disease_HALD.csv
HALD/   Gene_Gene_HALD.csv              HALD/   Mutation_Gene_HALD.csv
HALD/   Gene_Mutation_HALD.csv          HALD/   Mutation_Mutation_HALD.csv
```



---
## 0 · Configuration — edit ONLY these two lines

In [14]:
import os
import pandas as pd
import numpy as np
from glob import glob

# ─────────────────────────────────────────────────────────────────────────────
# USER CONFIGURATION
# BASE_PATH : root folder containing all raw input data
# OUT_PATH  : folder where all processed CSVs will be saved
# ─────────────────────────────────────────────────────────────────────────────
BASE_PATH = "/storage/Arushi/090526_EvoAge/kg_formation/data_collection/"
OUT_PATH  = "/storage/Arushi/090526_EvoAge/kg_formation/processed_data/hald/"

# ── Derived input paths ───────────────────────────────────────────────────────
PUBCHEM_SYN_PATH   = os.path.join(BASE_PATH, "databases_for_mapping/pubchem/CID-Synonym-filtered")
PUBCHEM_PKL_PATH   = os.path.join(BASE_PATH, "databases_for_mapping/pubchem/combined_df.pkl")
DRUGBANK_PATH      = os.path.join(BASE_PATH, "databases_for_mapping/drugbank/drugbank_all.csv")
DO_PATH            = os.path.join(BASE_PATH, "databases_for_mapping/DO/All_DO_ref_files.csv")
MESH2DOID_PATH     = os.path.join(BASE_PATH, "databases_for_mapping/DO/HumanDiseaseOntology-main/DOreports/MESHinDO.tsv")
MEDGEN_PATH        = os.path.join(BASE_PATH, "databases_for_mapping/MESH/MeSH/MedGenIDMappings.txt")
MESH_COMBINED_PATH = os.path.join(BASE_PATH, "databases_for_mapping/MESH/MESH_Combined.csv")
MESH_SUPP_PATH     = os.path.join(BASE_PATH, "databases_for_mapping/MESH/Mesh_Supplementary_Info.csv")
ENS2NCBI_PATH      = os.path.join(BASE_PATH, "databases_for_mapping/ENSEMBL/ENSEMBLE_ID_2_NCBI_Gene_SYM.csv")
NCBI_HUMAN_PATH    = os.path.join(BASE_PATH, "databases_for_mapping/ncbi/Homo_sapiens.gene_info")
HALD_ENTITIES_PATH = os.path.join(BASE_PATH, "hald/Entities.csv")
HALD_ROLES_PATH    = os.path.join(BASE_PATH, "hald/Roles.csv")

os.makedirs(OUT_PATH, exist_ok=True)

# Standard KG schema column order applied to all 16 edge type outputs
DESIRED_COLS = [
    "Head", "Relation", "Tail", "Head_type", "Relation_type", "Tail_type",
    "Source", "KG_Source", "Species",
    "Head_Alt_ID", "Head_detail_name", "Head_SMILES_name",
    "Tail_DO_Alt_ids", "Tail_IUPAC", "Tail_SMILES_name",
    "Tail_detail_name", "Head_ID_IS", "Tail_ID_IS",
    "Kg_index", "Kg", "Change"
]

print("Paths configured successfully.")
print(f"  Input  root : {BASE_PATH}")
print(f"  Output dir  : {OUT_PATH}")

Paths configured successfully.
  Input  root : /storage/Arushi/090526_EvoAge/kg_formation/data_collection/
  Output dir  : /storage/Arushi/090526_EvoAge/kg_formation/processed_data/hald/


---
## 1 · Load all reference dictionaries

In [15]:
# ── 1a. PubChem synonym → CID ─────────────────────────────────────────────────
Pubchem_Syn_fil = pd.read_csv(PUBCHEM_SYN_PATH, sep='\t', header=None)
Pubchem_Syn_fil_dict = dict(zip(Pubchem_Syn_fil[1], Pubchem_Syn_fil[0]))
Pubchem_Syn_fil_dict_lower = {str(k).lower(): v for k, v in Pubchem_Syn_fil_dict.items()}
print(f"PubChem synonyms: {len(Pubchem_Syn_fil_dict_lower):,}")

PubChem synonyms: 102,877,659


In [16]:
# ── 1b. PubChem CID → IUPAC name and SMILES ──────────────────────────────────
Pubchem = pd.read_pickle(PUBCHEM_PKL_PATH)
Pubchem_CID_Smile_Dict = dict(zip(Pubchem['PUBCHEM_COMPOUND_CID'], Pubchem['PUBCHEM_SMILES']))
Pubchem_IUPAC_CID_Dict = dict(zip(Pubchem['PUBCHEM_COMPOUND_CID'], Pubchem['PUBCHEM_IUPAC_NAME']))
del Pubchem  # free memory — large pickle no longer needed
print(f"PubChem CID -> IUPAC: {len(Pubchem_IUPAC_CID_Dict):,}")

PubChem CID -> IUPAC: 119,177,440


In [17]:
# ── 1c. DrugBank name <-> ID ──────────────────────────────────────────────────
DB = pd.read_csv(DRUGBANK_PATH)
DB_name_dict    = dict(zip(DB['name'],        DB['drugbank_id']))  # name -> DB ID
DB_name_ID_dict = dict(zip(DB['drugbank_id'], DB['name']))         # DB ID -> name
print(f"DrugBank entries: {len(DB_name_dict):,}")

DrugBank entries: 16,575


In [18]:
# ── 1d. Disease Ontology (DO) ─────────────────────────────────────────────────
DO_All_File = pd.read_csv(DO_PATH)
DOID_disname_dict      = dict(zip(DO_All_File['id'],    DO_All_File['label']))
DOID_disAltID_dict     = dict(zip(DO_All_File['id'],    DO_All_File['xrefs']))
DOID_disname_DOID_dict = dict(zip(DO_All_File['label'], DO_All_File['id']))
print(f"DO entries: {len(DOID_disname_DOID_dict):,}")

DO entries: 11,787


In [19]:
# ── 1e. MeSH disease lookups ──────────────────────────────────────────────────
# MESH_dict: Name -> ID (resolves disease names that don't match DOID)
MESH = pd.read_csv(MESH_COMBINED_PATH)
MESH_dict = dict(zip(MESH['Name'], MESH['ID']))
print(f"MeSH combined: {len(MESH_dict):,}")

Mesh_supp = pd.read_csv(MESH_SUPP_PATH)
Mesh_supp_dict = dict(zip(Mesh_supp['ID'], Mesh_supp['Name']))  # ID -> Name
print(f"MeSH supplementary: {len(Mesh_supp_dict):,}")

# MeSH -> DOID cross-reference (exploded from comma-separated xrefs)
Mesh2DOID = pd.read_csv(MESH2DOID_PATH, sep='\t')
Mesh2DOID.rename(columns={'ID': 'id', 'LABEL': 'label'}, inplace=True)
Mesh2DOID['xrefs'] = Mesh2DOID['xrefs'].str.split(', ')
Mesh2DOID = Mesh2DOID.explode('xrefs')
Mesh2DOID_dict = dict(zip(Mesh2DOID['xrefs'], Mesh2DOID['id']))
print(f"MeSH -> DOID: {len(Mesh2DOID_dict):,}")

# MedGen CUI mappings
MedGen_CUID_Source_ID = pd.read_csv(MEDGEN_PATH, sep='\t')
MedGen_CUID_Source_ID = MedGen_CUID_Source_ID.drop_duplicates(subset=['source_id','source'], keep='first')
MedGen_MeshID_name_dict = dict(zip(MedGen_CUID_Source_ID['source_id'], MedGen_CUID_Source_ID['pref_name']))

MeSH combined: 38,520
MeSH supplementary: 324,150
MeSH -> DOID: 3,687


In [20]:
# ── 1f. NCBI Human gene_info ──────────────────────────────────────────────────
ENS_2NCBI = pd.read_csv(ENS2NCBI_PATH)
ENS_2NCBI = ENS_2NCBI[~ENS_2NCBI['name'].isna()]
NCBI_2ENS__dict = dict(zip(ENS_2NCBI['name'], ENS_2NCBI['initial_alias']))
del ENS_2NCBI

NCBI_ALL_GENE = pd.read_csv(NCBI_HUMAN_PATH, sep='\t')
NCBI_ALL_GENE['ENSEMBLE_ID']  = NCBI_ALL_GENE['Symbol'].map(NCBI_2ENS__dict)
NCBI_ALL_GENEname_dict        = dict(zip(NCBI_ALL_GENE['GeneID'],  NCBI_ALL_GENE['description']))
NCBI_ALL_GENEIDname_dict      = dict(zip(NCBI_ALL_GENE['GeneID'],  NCBI_ALL_GENE['Symbol']))
NCBI_Synbol_GENEname_dict     = dict(zip(NCBI_ALL_GENE['Symbol'],  NCBI_ALL_GENE['description']))
print(f"NCBI Human genes: {len(NCBI_Synbol_GENEname_dict):,}")

NCBI Human genes: 193,352


---
## 2 · Load HALD entities and triples

In [21]:
# ── 2a. Load entity metadata: entity:ID, name, type ──────────────────────────
file = pd.read_csv(HALD_ENTITIES_PATH)
id_name_dict = dict(zip(file['entity:ID'], file['name']))
id_type_dict = dict(zip(file['entity:ID'], file['type']))
print(f"HALD entities: {len(file):,}")
print("Entity types:", set(file['type']))

HALD entities: 6,906
Entity types: {'Lipid', 'Gene', 'Pharmaceutical Preparations', 'Lipid;Peptide', 'Carbohydrate;Lipid', 'Carbohydrate', 'Disease', 'Lipid;Pharmaceutical Preparations', 'Peptide', 'Peptide;Protein', 'Lipid;Peptide;Protein', 'Mutation', 'Peptide;Toxin', 'Toxin', 'Carbohydrate;Peptide', 'RNA', 'Protein', 'Carbohydrate;Lipid;Toxin'}


In [22]:
# ── 2b. Load triples and map internal IDs -> names and types ─────────────────
triple_file = pd.read_csv(HALD_ROLES_PATH)

triple_file[':START_NAME'] = triple_file[':START_ID'].map(id_name_dict)
triple_file[':END_Name']   = triple_file[':END_ID'].map(id_name_dict)
triple_file[':START_TYPE'] = triple_file[':START_ID'].map(id_type_dict)
triple_file[':END_TYPE']   = triple_file[':END_ID'].map(id_type_dict)

triple_file = triple_file.rename(columns={
    ':START_NAME': 'Head',
    ':END_Name':   'Tail',
    'relation':    'Relation_type',
    ':START_ID':   'Head_Detail',
    ':END_ID':     'Tail_Detail',
    ':START_TYPE': 'Head_type',
    ':END_TYPE':   'Tail_type'
})

triple_file['Relation_Source'] = triple_file['Relation_type'] + '__' + triple_file['method'] + '_HALD'
triple_file['KG_Source'] = 'HALD_KG'
triple_file['Relation'] = triple_file['Head_type'] + '_' + triple_file['Tail_type']

desired_order = [
    'Head','Relation','Tail','Head_type','Tail_type','Relation_type',
    'Head_Detail','Tail_Detail','Relation_Source','KG_Source'
]
triple_file = triple_file[desired_order]
print(f"HALD triples: {len(triple_file):,}")
print("Raw Head types:", set(triple_file['Head_type']))

HALD triples: 115,514
Raw Head types: {'Lipid', 'Gene', 'Pharmaceutical Preparations', 'Lipid;Peptide', 'Carbohydrate;Lipid', 'Carbohydrate', 'Disease', 'Lipid;Pharmaceutical Preparations', 'Peptide', 'Peptide;Protein', 'Lipid;Peptide;Protein', 'Mutation', 'Peptide;Toxin', 'Toxin', 'Carbohydrate;Peptide', 'RNA', 'Protein', 'Carbohydrate;Lipid;Toxin'}


In [23]:
# ── 2c. Normalise entity types ────────────────────────────────────────────────
# Carbohydrate, Lipid, Peptide, Protein -> ChemicalEntity
# RNA -> Gene
chemical_keywords = ['Carbohydrate', 'Lipid', 'Peptide', 'Protein']

for col in ['Head_type', 'Tail_type']:
    triple_file[col] = triple_file[col].apply(
        lambda x: 'ChemicalEntity' if any(c in str(x) for c in chemical_keywords) else x
    )
    triple_file[col] = triple_file[col].replace('RNA', 'Gene')

# Remove Toxin and Pharmaceutical Preparations — no canonical ontology mapping
before = len(triple_file)
triple_file = triple_file[
    ~triple_file['Head_type'].isin(['Toxin','Pharmaceutical Preparations']) &
    ~triple_file['Tail_type'].isin(['Toxin','Pharmaceutical Preparations'])
]
print(f"After removing Toxin/PharmPrep: {len(triple_file):,} (removed {before-len(triple_file):,})")

# Rebuild Relation from normalised types
triple_file['Relation'] = triple_file['Head_type'] + '_' + triple_file['Tail_type']
print("Relation types:", sorted(set(triple_file['Relation'])))

After removing Toxin/PharmPrep: 115,465 (removed 49)
Relation types: ['ChemicalEntity_ChemicalEntity', 'ChemicalEntity_Disease', 'ChemicalEntity_Gene', 'ChemicalEntity_Mutation', 'Disease_ChemicalEntity', 'Disease_Disease', 'Disease_Gene', 'Disease_Mutation', 'Gene_ChemicalEntity', 'Gene_Disease', 'Gene_Gene', 'Gene_Mutation', 'Mutation_ChemicalEntity', 'Mutation_Disease', 'Mutation_Gene', 'Mutation_Mutation']


---
## 3 · Helper functions

The original repeated identical Chemical/Disease ID resolution blocks in every section.
Extracted into reusable functions here to eliminate duplication.

In [28]:
def resolve_chemical_head(df):
    """
    Resolve Head chemical name -> canonical ID (PubChem CID preferred, DrugBank fallback).
      1. name -> DrugBank ID (if found)
      2. DrugBank ID / original name -> PubChem CID (case-insensitive synonym lookup)
      3. Strip float suffix
      4. Annotate IUPAC name and SMILES
      5. Set Head_ID_IS namespace tag
    """
    df = df.copy()
    df['_head_tmp'] = df['Head'].map(DB_name_dict).fillna(df['Head'])
    df['Head'] = df['_head_tmp'].str.lower().map(Pubchem_Syn_fil_dict_lower).fillna(df['_head_tmp'])
    df['Head'] = df['Head'].astype(str).str.replace(r'\.0$', '', regex=True)
    df.drop(columns=['_head_tmp'], inplace=True)
    df['Head_detail_name'] = (df['Head'].map(Pubchem_IUPAC_CID_Dict)
                              .fillna(df['Head'].map(DB_name_ID_dict)).fillna(df['Head']))
    df['Head_SMILES_name'] = (df['Head'].map(Pubchem_CID_Smile_Dict)
                              .fillna(df['Head'].map(DB_name_ID_dict)).fillna(df['Head']))
    df['Head_ID_IS'] = np.where(
        df['Head'].str.startswith('DB', na=False), 'Drugbank',
        np.where(df['Head'].str.isnumeric(), 'Pubchem', np.nan))
    return df


def resolve_chemical_head(df):
    """
    Resolve Head chemical name -> canonical ID (PubChem CID preferred, DrugBank fallback).
    """
    df = df.copy()
    df['_head_tmp'] = df['Head'].map(DB_name_dict).fillna(df['Head'])
    df['Head'] = df['_head_tmp'].str.lower().map(Pubchem_Syn_fil_dict_lower).fillna(df['_head_tmp'])
    df['Head'] = df['Head'].astype(str).str.replace(r'\.0$', '', regex=True)
    df.drop(columns=['_head_tmp'], inplace=True)
    df['Head_detail_name'] = (df['Head'].map(Pubchem_IUPAC_CID_Dict)
                              .fillna(df['Head'].map(DB_name_ID_dict)).fillna(df['Head']))
    df['Head_SMILES_name'] = (df['Head'].map(Pubchem_CID_Smile_Dict)
                              .fillna(df['Head'].map(DB_name_ID_dict)).fillna(df['Head']))
    # FIX: use None instead of np.nan — np.nan is float and cannot be mixed
    # with string values in numpy 2.0+ without a DTypePromotionError
    df['Head_ID_IS'] = np.where(
        df['Head'].str.startswith('DB', na=False), 'Drugbank',
        np.where(df['Head'].str.isnumeric(), 'Pubchem', None))
    return df


def resolve_chemical_tail(df):
    """Same resolution as resolve_chemical_head but applied to Tail."""
    df = df.copy()
    df['_tail_tmp'] = df['Tail'].map(DB_name_dict).fillna(df['Tail'])
    df['Tail'] = df['_tail_tmp'].str.lower().map(Pubchem_Syn_fil_dict_lower).fillna(df['_tail_tmp'])
    df['Tail'] = df['Tail'].astype(str).str.replace(r'\.0$', '', regex=True)
    df.drop(columns=['_tail_tmp'], inplace=True)
    df['Tail_detail_name'] = (df['Tail'].map(Pubchem_IUPAC_CID_Dict)
                              .fillna(df['Tail'].map(DB_name_ID_dict)).fillna(df['Tail']))
    df['Tail_SMILES_name'] = (df['Tail'].map(Pubchem_CID_Smile_Dict)
                              .fillna(df['Tail'].map(DB_name_ID_dict)).fillna(df['Tail']))
    df['Tail_ID_IS'] = np.where(
        df['Tail'].str.startswith('DB', na=False), 'Drugbank',
        np.where(df['Tail'].str.isnumeric(), 'Pubchem', None))  # None not np.nan
    return df


def resolve_disease(df, col):
    """
    Resolve disease name column to canonical DOID (preferred) or MeSH ID (fallback).
    """
    df = df.copy()
    detail_col = f'{col}_detail_name'
    id_is_col  = f'{col}_ID_IS'
    df[col] = df[col].replace(DISEASE_NORMALISE)
    df[detail_col] = (df[col].str.lower().map(DOID_lower)
                      .fillna(df[col].str.lower().map(MESH_lower)))
    before = len(df)
    df = df[~df[detail_col].isna()]
    print(f"  Disease {col} resolved: {len(df):,} kept / {before-len(df):,} dropped")
    df[[col, detail_col]] = df[[detail_col, col]]
    df[id_is_col] = np.where(
        df[col].isna(), None,                                    # None not np.nan
        np.where(df[col].str.startswith('DOID'), 'DOID', 'MESH'))
    return df


def select_cols(df):
    """Select and reorder to standard KG schema, skipping absent columns."""
    return df[[c for c in DESIRED_COLS if c in df.columns]]


def save(df, filename):
    """Save to OUT_PATH and print confirmation."""
    path = os.path.join(OUT_PATH, filename)
    df.to_csv(path, index=False)
    print(f"Saved {len(df):,} rows -> {path}")


print("Helper functions defined.")

Helper functions defined.


---
## 4 · Process and export each relation type

### 4a · ChemicalEntity_ChemicalEntity

In [29]:
C_C = triple_file[triple_file['Relation'] == 'ChemicalEntity_ChemicalEntity'].copy()

# Drop rows where Head or Tail name is missing before splitting
# (NaN values cause DTypePromotionError when exploding mixed str/float columns)
C_C = C_C[~C_C['Head'].isna()]
C_C = C_C[~C_C['Tail'].isna()]

# Some HALD chemical names are comma-separated lists — explode into individual rows
C_C['Head'] = C_C['Head'].str.split(r',\s*')
C_C['Tail'] = C_C['Tail'].str.split(r',\s*')
C_C = C_C.explode('Head').explode('Tail').reset_index(drop=True)

C_C = resolve_chemical_head(C_C)
C_C = resolve_chemical_tail(C_C)
C_C['KG_Source'] = 'HALD'
C_C.drop(columns=['Head_Detail', 'Tail_Detail'], inplace=True, errors='ignore')
C_C = select_cols(C_C)
save(C_C, 'Chemical_Chemical_HALD.csv')

Saved 1,045 rows -> /storage/Arushi/090526_EvoAge/kg_formation/processed_data/hald/Chemical_Chemical_HALD.csv


### 4b · ChemicalEntity_Disease

In [30]:
C_D = triple_file[triple_file['Relation'] == 'ChemicalEntity_Disease'].copy()
C_D = C_D[~C_D['Head'].isna() & ~C_D['Tail'].isna()]
C_D['Head'] = C_D['Head'].str.split(r',\s*')
C_D = C_D.explode('Head').reset_index(drop=True)
C_D = resolve_chemical_head(C_D)
C_D = resolve_disease(C_D, 'Tail')
C_D = select_cols(C_D)
save(C_D, 'Chemical_Disease_HALD.csv')

  Disease Tail resolved: 4,330 kept / 0 dropped
Saved 4,330 rows -> /storage/Arushi/090526_EvoAge/kg_formation/processed_data/hald/Chemical_Disease_HALD.csv


### 4c · ChemicalEntity_Gene

In [31]:
C_G = triple_file[triple_file['Relation'] == 'ChemicalEntity_Gene'].copy()
C_G = C_G[~C_G['Head'].isna() & ~C_G['Tail'].isna()]
C_G = resolve_chemical_head(C_G)
C_G = resolve_gene(C_G, 'Tail')
C_G = select_cols(C_G)
save(C_G, 'Chemical_Gene_HALD.csv')

  Gene Tail validated: 709 kept / 1 dropped
Saved 709 rows -> /storage/Arushi/090526_EvoAge/kg_formation/processed_data/hald/Chemical_Gene_HALD.csv


### 4d · ChemicalEntity_Mutation

In [32]:
C_M = triple_file[triple_file['Relation'] == 'ChemicalEntity_Mutation'].copy()
C_M = C_M[~C_M['Head'].isna()]
C_M = resolve_chemical_head(C_M)
C_M = select_cols(C_M)
save(C_M, 'Chemical_Mutation_HALD.csv')

Saved 29 rows -> /storage/Arushi/090526_EvoAge/kg_formation/processed_data/hald/Chemical_Mutation_HALD.csv


### 4e · Disease_ChemicalEntity

In [33]:
D_C = triple_file[triple_file['Relation'] == 'Disease_ChemicalEntity'].copy()
D_C = D_C[~D_C['Head'].isna() & ~D_C['Tail'].isna()]
D_C['Head'] = D_C['Head'].str.split(r',\s*')
D_C = D_C.explode('Head').reset_index(drop=True)
D_C = resolve_disease(D_C, 'Head')
D_C = resolve_chemical_tail(D_C)
D_C = select_cols(D_C)
save(D_C, 'Disease_Chemical_HALD.csv')

  Disease Head resolved: 2,706 kept / 393 dropped
Saved 2,706 rows -> /storage/Arushi/090526_EvoAge/kg_formation/processed_data/hald/Disease_Chemical_HALD.csv


### 4f · Disease_Disease

In [34]:
# ── Disease_Disease ───────────────────────────────────────────────────────────
D_D = triple_file[triple_file['Relation'] == 'Disease_Disease'].copy()
D_D = D_D[~D_D['Head'].isna() & ~D_D['Tail'].isna()]
D_D = resolve_disease(D_D, 'Head')
D_D = resolve_disease(D_D, 'Tail')
D_D = select_cols(D_D)
save(D_D, 'Disease_Disease_HALD.csv')

  Disease Head resolved: 80,171 kept / 15 dropped
  Disease Tail resolved: 80,162 kept / 9 dropped
Saved 80,162 rows -> /storage/Arushi/090526_EvoAge/kg_formation/processed_data/hald/Disease_Disease_HALD.csv


### 4g · Disease_Gene

In [35]:
# ── Disease_Gene ──────────────────────────────────────────────────────────────
D_G = triple_file[triple_file['Relation'] == 'Disease_Gene'].copy()
D_G = D_G[~D_G['Head'].isna() & ~D_G['Tail'].isna()]
D_G = resolve_disease(D_G, 'Head')
D_G = resolve_gene(D_G, 'Tail')
D_G = select_cols(D_G)
save(D_G, 'Disease_Gene_HALD.csv')

  Disease Head resolved: 6,832 kept / 0 dropped
  Gene Tail validated: 6,813 kept / 19 dropped
Saved 6,813 rows -> /storage/Arushi/090526_EvoAge/kg_formation/processed_data/hald/Disease_Gene_HALD.csv


### 4h · Disease_Mutation

In [36]:
# ── Disease_Mutation ──────────────────────────────────────────────────────────
D_M = triple_file[triple_file['Relation'] == 'Disease_Mutation'].copy()
D_M = D_M[~D_M['Head'].isna()]
D_M = resolve_disease(D_M, 'Head')
D_M = select_cols(D_M)
save(D_M, 'Disease_Mutation_HALD.csv')

  Disease Head resolved: 456 kept / 0 dropped
Saved 456 rows -> /storage/Arushi/090526_EvoAge/kg_formation/processed_data/hald/Disease_Mutation_HALD.csv


### 4i · Gene_ChemicalEntity

In [37]:
G_C = triple_file[triple_file['Relation'] == 'Gene_ChemicalEntity'].copy()
G_C = G_C[~G_C['Head'].isna() & ~G_C['Tail'].isna()]
G_C = resolve_gene(G_C, 'Head')
G_C = resolve_chemical_tail(G_C)
G_C = select_cols(G_C)
save(G_C, 'Gene_Chemical_HALD.csv')

  Gene Head validated: 820 kept / 1 dropped
Saved 820 rows -> /storage/Arushi/090526_EvoAge/kg_formation/processed_data/hald/Gene_Chemical_HALD.csv


### 4j · Gene_Disease

In [38]:
G_D = triple_file[triple_file['Relation'] == 'Gene_Disease'].copy()
G_D = G_D[~G_D['Head'].isna() & ~G_D['Tail'].isna()]
G_D = resolve_gene(G_D, 'Head')
G_D = resolve_disease(G_D, 'Tail')
G_D = select_cols(G_D)
save(G_D, 'Gene_Disease_HALD.csv')

  Gene Head validated: 12,892 kept / 25 dropped
  Disease Tail resolved: 12,892 kept / 0 dropped
Saved 12,892 rows -> /storage/Arushi/090526_EvoAge/kg_formation/processed_data/hald/Gene_Disease_HALD.csv


### 4k · Gene_Gene

In [39]:
G_G = triple_file[triple_file['Relation'] == 'Gene_Gene'].copy()
G_G = G_G[~G_G['Head'].isna() & ~G_G['Tail'].isna()]
G_G = resolve_gene(G_G, 'Head')
G_G = resolve_gene(G_G, 'Tail')
G_G = select_cols(G_G)
save(G_G, 'Gene_Gene_HALD.csv')

  Gene Head validated: 4,278 kept / 10 dropped
  Gene Tail validated: 4,266 kept / 12 dropped
Saved 4,266 rows -> /storage/Arushi/090526_EvoAge/kg_formation/processed_data/hald/Gene_Gene_HALD.csv


### 4l · Gene_Mutation

In [40]:
G_M = triple_file[triple_file['Relation'] == 'Gene_Mutation'].copy()
G_M = G_M[~G_M['Head'].isna()]
G_M = resolve_gene(G_M, 'Head')
G_M = select_cols(G_M)
save(G_M, 'Gene_Mutation_HALD.csv')

  Gene Head validated: 274 kept / 2 dropped
Saved 274 rows -> /storage/Arushi/090526_EvoAge/kg_formation/processed_data/hald/Gene_Mutation_HALD.csv


### 4m · Mutation_ChemicalEntity

In [41]:
M_C = triple_file[triple_file['Relation'] == 'Mutation_ChemicalEntity'].copy()
M_C = M_C[~M_C['Tail'].isna()]
M_C = resolve_chemical_tail(M_C)
M_C = select_cols(M_C)
save(M_C, 'Mutation_Chemical_HALD.csv')

Saved 27 rows -> /storage/Arushi/090526_EvoAge/kg_formation/processed_data/hald/Mutation_Chemical_HALD.csv


### 4n · Mutation_Disease

In [42]:
M_D = triple_file[triple_file['Relation'] == 'Mutation_Disease'].copy()
M_D = M_D[~M_D['Tail'].isna()]
M_D = resolve_disease(M_D, 'Tail')
M_D = select_cols(M_D)
save(M_D, 'Mutation_Disease_HALD.csv')

  Disease Tail resolved: 634 kept / 0 dropped
Saved 634 rows -> /storage/Arushi/090526_EvoAge/kg_formation/processed_data/hald/Mutation_Disease_HALD.csv


### 4o · Mutation_Gene

In [43]:
M_G = triple_file[triple_file['Relation'] == 'Mutation_Gene'].copy()
M_G = M_G[~M_G['Tail'].isna()]
M_G = resolve_gene(M_G, 'Tail')
M_G = select_cols(M_G)
save(M_G, 'Mutation_Gene_HALD.csv')

  Gene Tail validated: 313 kept / 4 dropped
Saved 313 rows -> /storage/Arushi/090526_EvoAge/kg_formation/processed_data/hald/Mutation_Gene_HALD.csv


### 4p · Mutation_Mutation

In [44]:
M_M = triple_file[triple_file['Relation'] == 'Mutation_Mutation'].copy()
M_M = select_cols(M_M)
save(M_M, 'Mutation_Mutation_HALD.csv')

Saved 165 rows -> /storage/Arushi/090526_EvoAge/kg_formation/processed_data/hald/Mutation_Mutation_HALD.csv


---
## 5 · Summary — all output files

In [45]:
# Updated to use OUT_PATH.
print(f"Output files under: {OUT_PATH}\n")
total = 0
summary_rows = []

for fname in sorted(os.listdir(OUT_PATH)):
    if not fname.endswith('.csv'):
        continue
    full = os.path.join(OUT_PATH, fname)
    df   = pd.read_csv(full)
    rows = len(df)
    size = os.path.getsize(full)
    total += 1
    print(f"  {fname:<45s}  {rows:>6,} rows  {size:>9,} bytes")
    summary_rows.append({
        'Filename':       fname,
        'no. of triples': rows,
        'Relation':       set(df['Relation'].dropna()) if 'Relation' in df.columns else '',
        'Head_ID_IS':     set(df['Head_ID_IS'].dropna()) if 'Head_ID_IS' in df.columns else '',
        'Tail_ID_IS':     set(df['Tail_ID_IS'].dropna()) if 'Tail_ID_IS' in df.columns else '',
    })

print(f"\nTotal: {total} files")
pd.DataFrame(summary_rows)

Output files under: /storage/Arushi/090526_EvoAge/kg_formation/processed_data/hald/

  Chemical_Chemical_HALD.csv                      1,045 rows    356,845 bytes
  Chemical_Disease_HALD.csv                       4,330 rows  1,065,468 bytes
  Chemical_Gene_HALD.csv                            709 rows    191,928 bytes
  Chemical_Mutation_HALD.csv                         29 rows      5,508 bytes
  Disease_Chemical_HALD.csv                       2,706 rows    707,195 bytes
  Disease_Disease_HALD.csv                       80,162 rows  8,832,264 bytes
  Disease_Gene_HALD.csv                           6,813 rows    765,416 bytes
  Disease_Mutation_HALD.csv                         456 rows     42,904 bytes
  Gene_Chemical_HALD.csv                            820 rows    215,678 bytes
  Gene_Disease_HALD.csv                          12,892 rows  1,453,656 bytes
  Gene_Gene_HALD.csv                              4,266 rows    506,633 bytes
  Gene_Mutation_HALD.csv                            274 r

,Filename,no. of triples,Relation,Head_ID_IS,Tail_ID_IS,KG_Source
0,Chemical_Chemical_HALD.csv,1045,{ChemicalEntity_ChemicalEntity},"{Drugbank, Pubchem}","{Drugbank, Pubchem}",{HALD}
1,Chemical_Disease_HALD.csv,4330,{ChemicalEntity_Disease},"{Drugbank, Pubchem}","{DOID, MESH}",{HALD_KG}
2,Chemical_Gene_HALD.csv,709,{ChemicalEntity_Gene},"{Drugbank, Pubchem}",{NCBI_ID},{HALD_KG}
3,Chemical_Mutation_HALD.csv,29,{ChemicalEntity_Mutation},"{Drugbank, Pubchem}",,{HALD_KG}
4,Disease_Chemical_HALD.csv,2706,{Disease_ChemicalEntity},"{DOID, MESH}","{Drugbank, Pubchem}",{HALD_KG}
5,Disease_Disease_HALD.csv,80162,{Disease_Disease},"{DOID, MESH}","{DOID, MESH}",{HALD_KG}
6,Disease_Gene_HALD.csv,6813,{Disease_Gene},"{DOID, MESH}",{NCBI_ID},{HALD_KG}
7,Disease_Mutation_HALD.csv,456,{Disease_Mutation},"{DOID, MESH}",,{HALD_KG}
8,Gene_Chemical_HALD.csv,820,{Gene_ChemicalEntity},{NCBI_ID},"{Drugbank, Pubchem}",{HALD_KG}
9,Gene_Disease_HALD.csv,12892,{Gene_Disease},{NCBI_ID},"{DOID, MESH}",{HALD_KG}
